# Assignment

## Data Preparation

### **Fixed Sequence Length (C = 30)**

After trimming, all sequences still had different lengths depending on the leght of the squat. Instead of using zero padding, every sequence was converted into a fixed sequence length of C = 30 frames by selecting equally spaced frames across the entire movement. This ensured that all videos had the same dimension while still preserving information from the full squat motion. Sequences shorter than 30 frames were excluded from the dataset.


### **Features**

For every selected frame, MediaPipe pose landmarks were used to extract:

* Head
* Left/right shoulder
* Left/right elbow
* Left/right hand
* Left/right hip
* Left/right knee
* Left/right foot

For each joint, the following features were stored:

* x-coordinate
* y-coordinate
* z-coordinate

This resulted in: $\quad 13 \text{ joints} \times 3 \text{ features} = 39 \text{ features per frame}$

After converting each sequence to $30$ frames: $\quad 30 \text{ frames} \times 39 \text{ features}$

Each video sequence was finally flattened into one row in a CSV file.


### **Target**

The target values were loaded from a separate CSV file containing:

* video filename
* exercise score

The original scores were scaled into the interval $0-4$ using MinMax scaling. Videos marked as not squat were manually removed before scaling to avoid affecting the score distribution.

## RNN models

We test the performance of different RNN models, one simple, one LSTM and one GRU.

### Data Prep

In [1]:
import numpy as np
import os
import torch
from torchinfo import summary
import data_functions as dfunc

folder_path = "../../MainProject/data/mediapipe_trimmed_world"
score_path = "../../MainProject/data/video_scores.csv"

device = torch.device("cpu")

random_state = 42

train_data, test_data, train_y, test_y = dfunc.load_folder_with_split(
    folder_path=folder_path,
    target_path=score_path,
    test_size=0.2,
    random_state=random_state,
    select_rows=True,
    num_rows=30)

X_test = dfunc.augument_data(test_data, mirror_axis=[[False, False, False]], rotations=[0])
Y_test = torch.tensor(test_y, dtype=torch.float32).reshape(-1, 1)

In [2]:
RNN_folder_path = "scoring_models/RNN_models"

RNNcheckpoint = torch.load(os.path.join(RNN_folder_path, "SimpleRNNModel_scoring_checkpoint.pth"), weights_only=False)
LSTMcheckpoint = torch.load(os.path.join(RNN_folder_path, "LSTMModel_scoring_checkpoint.pth"), weights_only=False)
GRUcheckpoint = torch.load(os.path.join(RNN_folder_path, "GRUModel_scoring_checkpoint.pth"), weights_only=False)

RNN_checkpoints = [RNNcheckpoint, LSTMcheckpoint, GRUcheckpoint]

RNN_models = []
for checkpoint in RNN_checkpoints:
    model = checkpoint["model_config"]["model_type"](
        checkpoint["model_config"]["input_size"],
        checkpoint["model_config"]["hidden_size"],
        checkpoint["model_config"]["depth"],
        checkpoint["model_config"]["num_of_classification_labels"],
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    RNN_models.append(model)

for i, model in enumerate(RNN_models):
    if model is not None:
        print(f"{model.to_string()} successfully loaded!")
        print(f"Parameters:")
        for parameter, value in RNN_checkpoints[i]["model_config"].items():
            print(f"\t{parameter}: {value}")
        print()

SimpleRNNModel successfully loaded!
Parameters:
	model_type: <class 'RNN_models.SimpleRNNModel'>
	classifier: False
	input_size: 39
	hidden_size: 32
	depth: 5
	num_of_classification_labels: 1
	lr: 0.0001
	loss_func: <class 'torch.nn.modules.loss.MSELoss'>
	optimizer: <class 'torch.optim.adam.Adam'>
	patience: 3
	epochs: 100

LSTMModel successfully loaded!
Parameters:
	model_type: <class 'RNN_models.LSTMModel'>
	classifier: False
	input_size: 39
	hidden_size: 32
	depth: 5
	num_of_classification_labels: 1
	lr: 0.0001
	loss_func: <class 'torch.nn.modules.loss.MSELoss'>
	optimizer: <class 'torch.optim.adam.Adam'>
	patience: 3
	epochs: 100

GRUModel successfully loaded!
Parameters:
	model_type: <class 'RNN_models.GRUModel'>
	classifier: False
	input_size: 39
	hidden_size: 32
	depth: 5
	num_of_classification_labels: 1
	lr: 0.0001
	loss_func: <class 'torch.nn.modules.loss.MSELoss'>
	optimizer: <class 'torch.optim.adam.Adam'>
	patience: 3
	epochs: 100



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

for i, model in enumerate(RNN_models):
    model.eval()
    with torch.no_grad():
        predictions = model(X_test).cpu().numpy()
        
        # Calculate metrics
        mse = mean_squared_error(Y_test, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(Y_test, predictions)
        r2 = r2_score(Y_test, predictions)
        
        # Print metrics
        print(f"\n{'='*50}")
        print(f"Model: {RNN_models[i].to_string()}")
        print(f"{'='*50}")
        print(f"MSE:  {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE:  {mae:.4f}")
        print(f"R²:   {r2:.4f}")
        
        # Create visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f"{RNN_models[i].to_string() if hasattr(RNN_models[i], 'to_string') else type(model).__name__} - Regression Evaluation", 
                     fontsize=14, fontweight='bold')
        
        # Plot 1: Actual vs Predicted
        axes[0].scatter(Y_test, predictions, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[0].plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2, label='Perfect Prediction')
        axes[0].set_xlabel('Actual Values')
        axes[0].set_ylabel('Predicted Values')
        axes[0].set_title(f'Actual vs Predicted (R² = {r2:.3f})')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Residuals
        residuals = Y_test - predictions
        axes[1].scatter(predictions, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
        axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
        axes[1].set_xlabel('Predicted Values')
        axes[1].set_ylabel('Residuals')
        axes[1].set_title(f'Residual Plot (MAE = {mae:.3f})')
        axes[1].grid(True, alpha=0.3)
        
        # Plot 3: Distribution of errors
        axes[2].hist(residuals, bins=30, alpha=0.7, edgecolor='black', color='steelblue')
        axes[2].axvline(x=0, color='r', linestyle='--', lw=2)
        axes[2].set_xlabel('Residual Value')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Distribution of Residuals')
        axes[2].grid(True, alpha=0.3, axis='y')
        
        # Add text box with metrics
        metrics_text = f'MSE: {mse:.4f}\nRMSE: {rmse:.4f}\nMAE: {mae:.4f}\nR²: {r2:.4f}'
        axes[2].text(0.95, 0.95, metrics_text, transform=axes[2].transAxes,
                    fontsize=9, verticalalignment='top', horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.tight_layout()
        plt.show()

NameError: name 'models' is not defined

In [ ]:
champion_dir = "recording_classifier_models/cnn_model"
metadata_dir = "recording_classifier_models/cnn_model"

model_path = os.path.join(champion_dir, "champion_model.pt")
info_path = os.path.join(metadata_dir, "champion_info.json")

state_dict = torch.load(model_path, map_location=device)

with open(info_path, "r") as f:
    champion_info = json.load(f)

best_config = champion_info["hyperparameters"]

print(f"Champion model configuration: {best_config}\n")

# Rebuild model
model = build_cnn_model(best_config, input_shape)
model.load_state_dict(state_dict)
model.to(device) 


# Summarize model
model_summary = summary(model, input_size=(1, 1, input_shape[0], input_shape[1]))

print(model_summary)


model.to(device)
model.eval()

with torch.no_grad():
    logits = model(x_test.to(device))
    probs = torch.sigmoid(logits).cpu().numpy().ravel()

y_true = y_test.cpu().numpy().ravel()

# Metrics
test_auc = roc_auc_score(y_true, probs)
preds = (probs >= 0.5).astype(int)

accuracy = accuracy_score(y_true, preds)
precision = precision_score(y_true, preds, zero_division=0)
recall = recall_score(y_true, preds, zero_division=0)

print("\nTEST EVALUATION CNN MODEL (Champion Model)")
print(f"AUC: {test_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ugly", "Good"])
disp.plot(cmap="Blues")
plt.title("Champion Model - Confusion Matrix")
plt.show()